In [1]:
#new trail

In [2]:
"""
================================================================================
COMET SCORING NOTEBOOK  (separate Kaggle notebook -- do NOT run Part 1 here)
================================================================================
Fixes the crash:  ModuleNotFoundError: No module named 'numpy.char'
                  Could not import module 'AutoModelForMaskedLM'

Root cause
  `pip install unbabel-comet` pins numpy<2 and DOWNGRADES numpy (2.x -> 1.26.4) while the
  notebook kernel already has numpy 2.x loaded in memory (and scipy/sklearn/torchmetrics are
  compiled for numpy 2). Half old / half new numpy in one process => the import chain
  (comet -> pytorch_lightning -> torchmetrics -> transformers -> sklearn -> scipy -> numpy) dies.
  transformers 5.x is also incompatible with COMET.

What this script does differently
  1. Pins numpy/scipy/pandas/torch/protobuf to the versions ALREADY installed (pip constraints),
     so nothing in the environment is downgraded.
  2. Installs unbabel-comet with --no-deps plus only the packages it really needs, and
     transformers 4.48.3 (works with COMET).
  3. Runs the scoring in a fresh subprocess, so whatever the kernel imported earlier cannot clash.
  4. Keeps the COMET model cache in /tmp so your "Save Version" output stays small.

How to use
  1. New Kaggle notebook. Settings: Accelerator = GPU T4 (REQUIRED, your last run had GPU off),
     Internet = ON.
  2. Attach the NEWEST output of your PS-CAB/CAPM notebook (the one containing pscab_q1/).
  3. Paste this whole file into ONE cell, run it in a FRESH session (do not run any other cell first).
  4. "Save Version". Attach THIS notebook's output in the next Part 1 + Part 4 session.
Resumable: segment scores already on disk (analysis/comet__<run>__<split>.npy) are skipped.
================================================================================
"""
# ----------------------------------------------------------------------------- settings
COMET_MODEL = "Unbabel/wmt22-comet-da"     # report this exact name in the paper
SPLITS = ("test", "extfull_")              # file-name prefixes (after "__") to score; add "val" for validation
BATCH = 32
TRANSFORMERS_PIN = "4.48.3"                # transformers 5.x breaks COMET

# ----------------------------------------------------------------------------- launcher (no heavy imports here)
import os, sys, glob, json, shutil, subprocess, time, importlib.metadata as md

TMP = "/tmp/comet_run"
os.makedirs(TMP, exist_ok=True)


def sh(cmd, check=True, env=None):
    """Run a command, show the tail of its output if it fails."""
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0 and check:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"command failed: {' '.join(cmd[:6])} ...")
    return r


# ---- GPU check (via nvidia-smi, so torch is not imported in this kernel)
gpu = sh(["nvidia-smi", "-L"], check=False)
HAS_GPU = gpu.returncode == 0 and "GPU" in gpu.stdout
print("[env] GPU:", gpu.stdout.strip() if HAS_GPU else "NONE")
if not HAS_GPU:
    print("[WARN] No GPU. Scoring thousands of segments with XLM-R-large on CPU is extremely slow.\n"
          "       Notebook Settings -> Accelerator -> GPU T4, then run again in a fresh session.")

# ---- constraints: freeze everything COMET must not be allowed to change
frozen = []
for pkg in ["numpy", "scipy", "pandas", "scikit-learn", "protobuf", "torch", "torchvision", "torchaudio",
            "pillow", "pyarrow"]:
    try:
        frozen.append(f"{pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        pass
cons = os.path.join(TMP, "constraints.txt")
open(cons, "w").write("\n".join(frozen) + "\n")
print("[env] frozen:", ", ".join(frozen))

PIP = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons]

print(f"[pip] transformers=={TRANSFORMERS_PIN} + COMET runtime dependencies ...")
sh(PIP + [f"transformers=={TRANSFORMERS_PIN}", "sentencepiece", "entmax", "jsonargparse", "sacrebleu",
          "pytorch-lightning", "torchmetrics", "huggingface_hub", "pyyaml", "psutil", "packaging"])
print("[pip] unbabel-comet (--no-deps, so numpy is NOT downgraded) ...")
sh(PIP + ["--no-deps", "unbabel-comet"])

# ---- locate + copy results (identical behaviour to the original script)
def newest_results():
    cands = []
    for depth in range(1, 6):
        cands += glob.glob("/kaggle/input/" + "/".join(["*"] * depth) + "/pscab_q1/RESULTS_MARKER.json")
    if not cands:
        raise FileNotFoundError("No pscab_q1/RESULTS_MARKER.json under /kaggle/input -- attach your notebook output.")

    def key(m):
        try:
            return json.load(open(m)).get("updated", "")
        except Exception:
            return ""
    return os.path.dirname(max(cands, key=key))


SRC = newest_results()
R = "/kaggle/working/pscab_q1"
if not os.path.exists(f"{R}/RESULTS_MARKER.json"):
    print(f"[copy] {SRC} -> {R}")
    shutil.copytree(SRC, R, dirs_exist_ok=True)
os.makedirs(f"{R}/analysis", exist_ok=True)
os.makedirs(f"{R}/tables", exist_ok=True)

# ----------------------------------------------------------------------------- worker (runs in a fresh interpreter)
WORKER = r'''
import os, sys, glob, json, time, traceback
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"     # COMET checkpoints are full pickles (torch>=2.6 default breaks them)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import importlib.metadata as md
import numpy as np
import pandas as pd
import torch

cfg = json.load(open(sys.argv[1]))
R, MODEL, SPLITS, BATCH = cfg["R"], cfg["model"], tuple(cfg["splits"]), int(cfg["batch"])

for p in ("numpy", "torch", "transformers", "pytorch-lightning", "torchmetrics", "unbabel-comet"):
    try:
        print(f"[ver] {p:18s} {md.version(p)}", flush=True)
    except Exception:
        print(f"[ver] {p:18s} MISSING", flush=True)

try:
    from comet import download_model, load_from_checkpoint
except Exception:
    traceback.print_exc()
    sys.exit("COMET import failed (see traceback above).")

gpus = 1 if torch.cuda.is_available() else 0
ckpt = download_model(MODEL, saving_directory="/tmp/comet_models")
model = load_from_checkpoint(ckpt)
print(f"[comet] {MODEL} loaded, gpus={gpus}", flush=True)

files = sorted(f for f in glob.glob(f"{R}/preds/*__*.csv")
               if any(os.path.basename(f).split("__", 1)[1].startswith(p) for p in SPLITS))
print(f"[comet] {len(files)} prediction files match {SPLITS}", flush=True)


def score(data):
    bs = BATCH
    while True:
        try:
            res = model.predict(data, batch_size=bs, gpus=gpus, progress_bar=False, num_workers=2)
            return np.asarray(res.scores, dtype=np.float64)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 2:
                bs //= 2
                torch.cuda.empty_cache()
                print(f"[comet] CUDA OOM -> retrying with batch_size={bs}", flush=True)
            else:
                raise


rows = []
for f in files:
    run, split = os.path.basename(f)[:-4].split("__", 1)
    out = f"{R}/analysis/comet__{run}__{split}.npy"
    if os.path.exists(out):
        seg = np.load(out)
    else:
        d = pd.read_csv(f, dtype=str, keep_default_na=False)
        missing = {"en", "prediction", "te"} - set(d.columns)
        if missing:
            print(f"[skip] {os.path.basename(f)} lacks columns {missing}", flush=True)
            continue
        data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(d["en"], d["prediction"], d["te"])]
        t0 = time.time()
        seg = score(data)
        np.save(out, seg)
        print(f"[comet] {run:28s} {split:28s} n={len(seg):5d}  COMET={100*seg.mean():.2f}  ({time.time()-t0:.0f}s)", flush=True)
    rows.append({"run": run, "split": split, "n": len(seg), "COMET_x100": round(100 * float(seg.mean()), 2)})

summ = pd.DataFrame(rows).sort_values(["split", "run"])
os.makedirs(f"{R}/tables", exist_ok=True)
summ.to_csv(f"{R}/tables/COMET_summary.csv", index=False)
with open(f"{R}/analysis/comet_info.json", "w") as fh:
    json.dump({"model": MODEL, "scored": time.strftime("%Y-%m-%d %H:%M:%S"),
               "comet_version": md.version("unbabel-comet"),
               "transformers_version": md.version("transformers"),
               "torch_version": torch.__version__}, fh, indent=2)
with open(f"{R}/RESULTS_MARKER.json", "w") as fh:          # newest marker -> Part 1 restores this output
    json.dump({"updated": time.strftime("%Y-%m-%d %H:%M:%S")}, fh)
print(summ.to_string(index=False), flush=True)
print("\nDone. Save Version, then attach this output in the next Part 1 + Part 4 session.", flush=True)
'''

worker_py = os.path.join(TMP, "comet_worker.py")
cfg_json = os.path.join(TMP, "cfg.json")
open(worker_py, "w").write(WORKER)
json.dump({"R": R, "model": COMET_MODEL, "splits": list(SPLITS), "batch": BATCH}, open(cfg_json, "w"))

env = dict(os.environ, PYTHONUNBUFFERED="1", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1",
           HF_HOME="/tmp/hf_cache", TRANSFORMERS_CACHE="/tmp/hf_cache", TOKENIZERS_PARALLELISM="false")

print("\n[run] starting scoring subprocess ...\n")
proc = subprocess.Popen([sys.executable, worker_py, cfg_json], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"COMET worker exited with code {proc.returncode}; see the output above.")

[env] GPU: GPU 0: Tesla T4 (UUID: GPU-03fb1704-4c3a-8f44-a062-f2ff4781b251)
GPU 1: Tesla T4 (UUID: GPU-de2c1c2c-b056-e964-40f2-b06927dc24e7)
[env] frozen: numpy==2.1.3, scipy==1.16.3, pandas==2.3.3, scikit-learn==1.6.1, protobuf==7.36.2, torch==2.11.0+cu130, torchvision==0.26.0+cu130, torchaudio==2.11.0+cu130, pillow==13.0.0, pyarrow==23.0.1
[pip] transformers==4.48.3 + COMET runtime dependencies ...
[pip] unbabel-comet (--no-deps, so numpy is NOT downgraded) ...
[copy] /kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1 -> /kaggle/working/pscab_q1

[run] starting scoring subprocess ...

[ver] numpy              2.1.3
[ver] torch              2.11.0+cu130
[ver] transformers       4.48.3
[ver] pytorch-lightning  2.6.6
[ver] torchmetrics       1.9.0
[ver] unbabel-comet      2.2.7
/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead